In [ ]:
# PUBLIC_MATERIALS_NOTEBOOK_SETUP
from pathlib import Path
from urllib.request import urlretrieve

try:
    import google.colab  # type: ignore  # noqa: F401
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

_MATERIALS_BASE = \"https://raw.githubusercontent.com/yonpak/ai-from-zero-to-builder-materials/main/labs/notebooks/tiny-llm/\"
if _IN_COLAB or not Path(\"tiny_llm.py\").exists():
    for _name in (\"tiny_llm.py\", \"tiny_corpus.txt\"):
        urlretrieve(_MATERIALS_BASE + _name, _name)


# Causal Self-Attention From Tensors

This notebook supports `l05-02-words-looking-at-words` and `l05-07-self-attention`.

The main invariant is simple: a token may attend to itself and earlier tokens, but not future tokens.

In [ ]:
import torch
from tiny_llm import causal_attention, future_attention_mass, set_seed

set_seed(7)

# One batch, four time steps, four features.
q = torch.eye(4).unsqueeze(0)
k = q.clone()
v = q.clone()

out, weights = causal_attention(q, k, v)
print(weights)
print("future attention mass:", future_attention_mass(weights))

In [ ]:
assert weights.shape == (1, 4, 4)
assert torch.allclose(weights.sum(dim=-1), torch.ones(1, 4))
assert future_attention_mass(weights) == 0.0

## Break it on purpose

Remove the causal mask from the helper implementation and rerun the checks.
`future_attention_mass(weights)` becomes greater than zero.

That failure is useful: it catches information leakage from the answer token into the prediction.

In [ ]:
from tiny_llm import TinyConfig, CausalSelfAttention

config = TinyConfig(vocab_size=20, block_size=8, n_embd=32, n_head=4, n_layer=1)
attention = CausalSelfAttention(config)
x = torch.randn(2, 5, config.n_embd)
y = attention(x)

print("input shape:", tuple(x.shape))
print("output shape:", tuple(y.shape))
assert y.shape == x.shape

## Validation

- Every attention row sums to approximately `1`.
- Probability above the causal diagonal is exactly `0`.
- Multi-head self-attention preserves `(batch, time, embedding)` shape.
- CPU is sufficient.